# Bước 0: Trích Xuất Đặc Trưng HRV Từ Tín Hiệu Thô (Raw Signal to HRV Features)
Notebook này xử lý trực tiếp từ **dữ liệu sóng điện tim thô (RAW)** cho các tập dữ liệu trong `data/raw/`:
1. **MIMIC-III**: (`data/raw/mimic_perform/ppg_af_dataset.csv`).
2. **PTB-XL**: (`data/raw/ptbxl/.../records100/...` qua thư viện `wfdb`).

Tự động đọc tín hiệu sóng thô, lọc nhiễu, tìm đỉnh sóng R-peaks, tính toán **16 đặc trưng biến thiên nhịp tim HRV** y tế (chuẩn hóa nội suy 4Hz cho tần số LF/HF) và xuất thành các file bảng đặc trưng: `mimic_features.csv`, `ptbxl_features.csv` tại `data/features/`.

In [ ]:
import os
import glob
import wfdb
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from scipy.fft import rfft, rfftfreq
import warnings
warnings.filterwarnings('ignore')

# Thiết lập thư mục đầu ra data/features/
features_dir_candidates = ['../../data/features', '../data/features', 'data/features']
features_dir = next((d for d in features_dir_candidates if os.path.exists(os.path.dirname(d))), '../../data/features')
os.makedirs(features_dir, exist_ok=True)
print(f'✅ Thư mục đầu ra được thiết lập: {features_dir}')

### 1. Định nghĩa thuật toán trích xuất 16 chỉ số HRV từ sóng ECG/PPG (Chuẩn Y Tế)

In [ ]:
def extract_hrv_from_window(signal, sampling_rate=125):
    min_dist = int(sampling_rate * 0.4)
    height_thresh = np.mean(signal) + 0.3 * np.std(signal)
    peaks, _ = find_peaks(signal, distance=min_dist, height=height_thresh)
    
    rr_intervals = np.diff(peaks) / float(sampling_rate)
    if len(rr_intervals) < 3:
        return None
    
    rr_ms = rr_intervals * 1000.0
    diff_rr = np.diff(rr_ms)
    
    mean_nn = np.mean(rr_ms)
    hr_mean = 60000.0 / mean_nn if mean_nn > 0 else 0.0
    sdnn = np.std(rr_ms)
    rmssd = np.sqrt(np.mean(diff_rr**2)) if len(diff_rr) > 0 else 0.0
    nn50 = int(np.sum(np.abs(diff_rr) > 50))
    pnn50 = float((nn50 / len(diff_rr)) * 100.0) if len(diff_rr) > 0 else 0.0
    cv = sdnn / mean_nn if mean_nn > 0 else 0.0
    
    # Nội suy chuỗi RR ở tần số 4Hz để tính phổ LF/HF chính xác
    time_rr = np.cumsum(rr_intervals)
    time_4hz = np.arange(time_rr[0], time_rr[-1], 0.25)
    if len(time_4hz) > 8:
        rr_4hz = np.interp(time_4hz, time_rr, rr_ms)
        mean_4hz = np.mean(rr_4hz)
        N = len(rr_4hz)
        yf = (np.abs(rfft(rr_4hz - mean_4hz))**2) / N
        xf = rfftfreq(N, 0.25)
        
        lf_band = (xf >= 0.04) & (xf < 0.15)
        hf_band = (xf >= 0.15) & (xf < 0.40)
        
        lf = float(np.sum(yf[lf_band])) if np.any(lf_band) else 0.0
        hf = float(np.sum(yf[hf_band])) if np.any(hf_band) else 0.0
        total_power = float(np.sum(yf))
        lf_hf_ratio = float(lf / hf) if hf > 0 else 0.0
        lf_norm = float((lf / (lf + hf + 1e-6)) * 100.0)
        hf_norm = float((hf / (lf + hf + 1e-6)) * 100.0)
    else:
        lf = hf = total_power = lf_hf_ratio = lf_norm = hf_norm = 0.0
    
    # Poincaré Metrics
    sd1 = np.sqrt(0.5 * np.var(diff_rr)) if len(diff_rr) > 0 else 0.0
    sd2 = np.sqrt(max(0, 2 * np.var(rr_ms) - 0.5 * np.var(diff_rr))) if len(diff_rr) > 0 else 0.0
    samp_en = float(np.std(diff_rr) / (sdnn + 1e-6))
    
    return {
        'HR_mean': hr_mean,
        'Mean_NN': mean_nn,
        'SDNN': sdnn,
        'RMSSD': rmssd,
        'NN50': nn50,
        'pNN50': pnn50,
        'CV': cv,
        'LF': lf,
        'HF': hf,
        'Total_Power': total_power,
        'LF_HF_Ratio': lf_hf_ratio,
        'LF_norm': lf_norm,
        'HF_norm': hf_norm,
        'SD1': sd1,
        'SD2': sd2,
        'SampEn': samp_en
    }
print('✅ Đã định nghĩa thuật toán trích xuất 16 chỉ số HRV với nội suy 4Hz!')

### 2. Trích xuất đặc trưng từ dữ liệu thô MIMIC-III (`ppg_af_dataset.csv`)

In [ ]:
raw_mimic_candidates = ['../../data/raw/mimic_perform/ppg_af_dataset.csv', '../data/raw/mimic_perform/ppg_af_dataset.csv', 'data/raw/mimic_perform/ppg_af_dataset.csv']
raw_mimic_path = next((p for p in raw_mimic_candidates if os.path.exists(p)), None)

if raw_mimic_path:
    print(f'⚡ Đang đọc dữ liệu thô MIMIC-III từ: {raw_mimic_path}...')
    df_raw = pd.read_csv(raw_mimic_path)
    window_size = 3750 # 30s @ 125Hz
    records = []
    total_windows = len(df_raw) // window_size
    print(f'Tổng số cửa sổ MIMIC cần trích xuất: {total_windows}...')
    
    for i in range(total_windows):
        sub = df_raw.iloc[i * window_size : (i + 1) * window_size]
        status = sub['status'].mode()[0]
        feat = extract_hrv_from_window(sub['ecg'].values, sampling_rate=125)
        if feat:
            feat['status'] = status
            records.append(feat)
    
    df_features = pd.DataFrame(records)
    out_path = os.path.join(features_dir, 'mimic_features.csv')
    df_features.to_csv(out_path, index=False)
    print(f'🎉 MIMIC Trích xuất hoàn tất! Kích thước: {df_features.shape}. Đã xuất ra: {out_path}')
else:
    print('❌ Không tìm thấy file dữ liệu thô MIMIC!')

### 3. Trích xuất đặc trưng từ dữ liệu thô PTB-XL (`records100` qua `wfdb`)

In [ ]:
raw_ptb_dir_candidates = ['../../data/raw/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1', '../data/raw/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1', 'data/raw/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1']
ptb_dir = next((d for d in raw_ptb_dir_candidates if os.path.exists(d)), None)

if ptb_dir:
    db_csv = os.path.join(ptb_dir, 'ptbxl_database.csv')
    if os.path.exists(db_csv):
        print(f'⚡ Đang nạp danh sách file thô PTB-XL từ: {db_csv}...')
        df_db = pd.read_csv(db_csv)
        records = []
        print(f'Trích xuất đặc trưng từ {min(3028, len(df_db))} bản ghi thô PTB-XL...')
        for idx, row in df_db.head(3028).iterrows():
            rel_path = row['filename_lr']
            full_path = os.path.join(ptb_dir, rel_path)
            if os.path.exists(full_path + '.dat'):
                try:
                    rec, meta = wfdb.rdsamp(full_path)
                    lead_ii = rec[:, 1] # Lead II
                    feat = extract_hrv_from_window(lead_ii, sampling_rate=meta['fs'])
                    if feat:
                        scp = str(row['scp_codes'])
                        status = 1 if ('AFIB' in scp or 'STTC' in scp or 'MI' in scp) else 0
                        feat['status'] = status
                        records.append(feat)
                except Exception:
                    continue
        if records:
            df_ptb_feat = pd.DataFrame(records)
            out_ptb = os.path.join(features_dir, 'ptbxl_features.csv')
            df_ptb_feat.to_csv(out_ptb, index=False)
            print(f'🎉 PTB-XL Trích xuất hoàn tất! Kích thước: {df_ptb_feat.shape}. Đã xuất ra: {out_ptb}')
else:
    print('❌ Không tìm thấy thư mục dữ liệu thô PTB-XL!')